In [ ]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import warnings
warnings.filterwarnings("ignore")

In [ ]:
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_excel(
    "/content/drive/MyDrive/Internship project/04_ground_truth_dataset.xlsx"
)

print(df.shape)

df.head()

(1206, 9)


,Message_ID,Person,Chat_Name,Timestamp,Sender,Message,Risk_Label,Risk_Category,Confidence
0,1,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:15:00,Vishnu,"Da, report kandille?",Normal,NaN,High
1,2,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:17:00,You,Kandu. Ellam okay alle?,Normal,NaN,High
2,3,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:18:00,Vishnu,Mostly okay. Pakshe aa last item kurachu stran...,Suspicious,Coded Language,Medium
3,4,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:20:00,You,Entha issue?,Normal,NaN,High
4,5,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:21:00,Vishnu,Numbers match cheyyunnilla. Randu places il di...,Suspicious,Coded Language,Medium


In [ ]:
df.isnull().sum()

,0
Message_ID,0
Person,0
Chat_Name,0
Timestamp,0
Sender,0
Message,0
Risk_Label,0
Risk_Category,952
Confidence,0


In [ ]:
df["Risk_Category"] = df["Risk_Category"].fillna("None")

In [ ]:
print(df.isnull().sum())

Message_ID       0
Person           0
Chat_Name        0
Timestamp        0
Sender           0
Message          0
Risk_Label       0
Risk_Category    0
Confidence       0
dtype: int64


In [ ]:
df["Processed_Message"] = df["Message"].astype(str)

In [ ]:
df["Processed_Message"] = (
    df["Processed_Message"]
    .str.lower()
)

In [ ]:
df["Processed_Message"] = (
    df["Processed_Message"]
    .str.replace(
        r"http\S+|www\S+",
        "",
        regex=True
    )
)

In [ ]:
df["Processed_Message"] = (
    df["Processed_Message"]
    .str.replace(
        r"<media omitted>",
        "",
        case=False,
        regex=True
    )
)

In [ ]:
df["Processed_Message"] = (
    df["Processed_Message"]
    .str.replace(
        r"[^\w\s]",
        " ",
        regex=True
    )
)

In [ ]:
df["Processed_Message"] = (
    df["Processed_Message"]
    .str.replace(
        r"\d{1,2}/\d{1,2}/\d{2},?\s+\d{1,2}:\d{2}\s*(am|pm)?",
        "",
        case=False,
        regex=True
    )
)

In [ ]:
df["Processed_Message"] = (
    df["Processed_Message"]
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)

In [ ]:
emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "]+",
    flags=re.UNICODE
)

df["Processed_Message"] = df["Processed_Message"].apply(
    lambda x: emoji_pattern.sub("", x)
)

In [ ]:
df["Processed_Message"] = (
    df["Processed_Message"]
    .str.replace(
        r"[^\w\s\u0D00-\u0D7F]",
        " ",
        regex=True
    )
)

In [ ]:
df["Processed_Message"] = (
    df["Processed_Message"]
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)

In [ ]:
nltk.download("punkt_tab")
df["Tokens"] = df["Processed_Message"].apply(word_tokenize)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
stop_words = set(stopwords.words("english"))

df["Tokens"] = df["Tokens"].apply(
    lambda words: [
        word
        for word in words
        if word not in stop_words
    ]
)

In [ ]:
lemmatizer = WordNetLemmatizer()

df["Tokens"] = df["Tokens"].apply(
    lambda words: [
        lemmatizer.lemmatize(word)
        for word in words
    ]
)

In [ ]:
df["Processed_Message"] = (
    df["Tokens"]
    .str.join(" ")
)

In [ ]:
df[
    [
        "Message",
        "Processed_Message",
        "Tokens"
    ]
].head(20)

,Message,Processed_Message,Tokens
0,"Da, report kandille?",da report kandille,"[da, report, kandille]"
1,Kandu. Ellam okay alle?,kandu ellam okay alle,"[kandu, ellam, okay, alle]"
2,Mostly okay. Pakshe aa last item kurachu stran...,mostly okay pakshe aa last item kurachu strang...,"[mostly, okay, pakshe, aa, last, item, kurachu..."
3,Entha issue?,entha issue,"[entha, issue]"
4,Numbers match cheyyunnilla. Randu places il di...,number match cheyyunnilla randu place il diffe...,"[number, match, cheyyunnilla, randu, place, il..."
5,Ath verify cheytho?,ath verify cheytho,"[ath, verify, cheytho]"
6,Cheythu. Athanu njan parayunne.,cheythu athanu njan parayunne,"[cheythu, athanu, njan, parayunne]"
7,Text il discuss cheyyanda. Evening vilikkam.,text il discus cheyyanda evening vilikkam,"[text, il, discus, cheyyanda, evening, vilikkam]"
8,Sheri.,sheri,[sheri]
9,Da innale paranja matter orthu?,da innale paranja matter orthu,"[da, innale, paranja, matter, orthu]"


In [ ]:
df.to_csv(
    "/content/drive/MyDrive/Internship project/05_processed_text_dataset.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!
